In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from src.utils.pipeline import load_all_snapshots

df = load_all_snapshots()

bb = df[df["launch_speed"].notna() & df["launch_angle"].notna()]
print(f"batted balls with measurements: {len(bb):,}")
print()
print(df["launch_speed_angle"].value_counts(dropna=False).sort_index())

batted balls with measurements: 237,198

launch_speed_angle
1         5509
2        38285
3        32269
4        30082
5         7761
6         9698
<NA>    587028
Name: count, dtype: Int64


In [2]:
summary = (
    bb.groupby("launch_speed_angle")
    .agg(
        n=("launch_speed", "size"),
        ev_mean=("launch_speed", "mean"),
        ev_min=("launch_speed", "min"),
        la_mean=("launch_angle", "mean"),
        la_min=("launch_angle", "min"),
        la_max=("launch_angle", "max"),
        xwoba=("estimated_woba_using_speedangle", "mean"),
    )
    .round(2)
)
print(summary.to_string())

                        n  ev_mean  ev_min  la_mean  la_min  la_max  xwoba
launch_speed_angle                                                        
1                    5509    47.73     8.8   -16.12     -90      89   0.18
2                   38285    85.46    59.5    -14.8     -88      22   0.18
3                   32269    85.96    59.5    46.07      21      90   0.08
4                   30082    93.32    59.6    11.45      -6      41   0.63
5                    7761   101.19    94.5    23.58       2      47   0.59
6                    9698   104.69    97.5    26.13       4      47   1.23


In [3]:
BARREL_BANDS = {   # exit velocity -> (la_min, la_max), from MLB glossary
    98:  (26, 30),
    99:  (25, 31),
    100: (24, 33),
    116: (8, 50),
}

barrel_col = summary["xwoba"].idxmax()   # whichever code has the top xwOBA
print("presumed barrel code:", barrel_col)

for ev, (lo, hi) in BARREL_BANDS.items():
    band = bb[(bb["launch_speed"] >= ev) & (bb["launch_speed"] < ev + 1)]
    if len(band) == 0:
        continue
    stated = band["launch_angle"].between(lo, hi)
    statcast = band["launch_speed_angle"] == barrel_col
    agree = (stated == statcast).mean()
    print(f"EV {ev}: n={len(band):5d}  glossary band {lo}-{hi}  "
          f"agreement with Statcast: {agree:.1%}")

presumed barrel code: 6
EV 98: n= 4646  glossary band 26-30  agreement with Statcast: 98.3%
EV 99: n= 4635  glossary band 25-31  agreement with Statcast: 97.8%
EV 100: n= 4407  glossary band 24-33  agreement with Statcast: 98.5%
EV 116: n=   35  glossary band 8-50  agreement with Statcast: 100.0%


In [4]:
HARD_HIT_MPH = 95
SWEET_SPOT_DEG = (8, 32)

bb = bb.copy()
bb["is_hard_hit"] = bb["launch_speed"] >= HARD_HIT_MPH
bb["is_sweet_spot"] = bb["launch_angle"].between(*SWEET_SPOT_DEG)
bb["is_barrel"] = bb["launch_speed_angle"] == barrel_col

print(f"batted balls: {len(bb):,}")
print(f"Barrel%:      {bb['is_barrel'].mean():.1%}")
print(f"HardHit%:     {bb['is_hard_hit'].mean():.1%}")
print(f"SweetSpot%:   {bb['is_sweet_spot'].mean():.1%}")
print(f"avg EV:       {bb['launch_speed'].mean():.1f} mph")
print(f"avg LA:       {bb['launch_angle'].mean():.1f} deg")
print()
print("all barrels are in the sweet spot:",
      (bb.loc[bb['is_barrel'], 'is_sweet_spot']).all())

batted balls: 237,198
Barrel%:      7.8%
HardHit%:     23.8%
SweetSpot%:   29.9%
avg EV:       82.5 mph
avg LA:       17.5 deg

all barrels are in the sweet spot: False


In [5]:
print("rows with launch_speed & launch_angle:", len(bb))
print("rows with launch_speed_angle:", bb["launch_speed_angle"].notna().sum())
print()
no_lsa = bb[bb["launch_speed_angle"].isna()]
print(f"missing launch_speed_angle: {len(no_lsa):,}")
print(no_lsa["description"].value_counts().head())
print()
print("EV of those rows:")
print(no_lsa["launch_speed"].describe().round(1))

rows with launch_speed & launch_angle: 237198
rows with launch_speed_angle: 123604

missing launch_speed_angle: 113,594
description
foul             113588
hit_into_play         6
Name: count, dtype: int64

EV of those rows:
count    113594.0
mean         76.2
std          12.8
min           3.0
25%          69.9
50%          76.2
75%          82.9
max         117.4
Name: launch_speed, dtype: Float64


In [6]:
bbe = df[
    (df["description"] == "hit_into_play")
    & df["launch_speed"].notna()
    & df["launch_angle"].notna()
].copy()

print(f"batted ball events: {len(bbe):,}")

bbe["is_hard_hit"] = bbe["launch_speed"] >= 95
bbe["is_sweet_spot"] = bbe["launch_angle"].between(8, 32)
bbe["is_barrel"] = bbe["launch_speed_angle"] == 6

print(f"Barrel%:    {bbe['is_barrel'].mean():.1%}")
print(f"HardHit%:   {bbe['is_hard_hit'].mean():.1%}")
print(f"SweetSpot%: {bbe['is_sweet_spot'].mean():.1%}")
print(f"avg EV:     {bbe['launch_speed'].mean():.1f} mph")
print(f"avg LA:     {bbe['launch_angle'].mean():.1f} deg")
print()
print("all barrels in sweet spot:", bbe.loc[bbe['is_barrel'], 'is_sweet_spot'].all())

batted ball events: 123,610
Barrel%:    7.8%
HardHit%:   39.0%
SweetSpot%: 35.2%
avg EV:     88.3 mph
avg LA:     13.0 deg

all barrels in sweet spot: False


In [7]:
outside = bbe[bbe["is_barrel"] & ~bbe["is_sweet_spot"]]
print(f"barrels outside sweet spot: {len(outside):,} of {bbe['is_barrel'].sum():,}")
print(outside[["launch_speed", "launch_angle"]].describe().round(1))

barrels outside sweet spot: 1,306 of 9,698
       launch_speed  launch_angle
count        1306.0        1306.0
mean          104.7          35.2
std             3.0           3.0
min            99.5           4.0
25%           102.5          33.0
50%           104.3          35.0
75%           106.4          37.0
max           120.4          47.0
